# Patient extraction · top/bottom-decile transferred-XGBoost risk patients and their pathology reports

Data request (Elio). At each landmark (ADT +0 / +90 / +180 days) the platinum-trained federated XGBoost model, refit
**without Dana-Farber** and scored on the local DFCI ADT cohort in notebook 04, gives each patient a risk score. This
notebook takes the patients in the **top or bottom decile** of that score at **any** landmark and writes:

| File | Content |
|---|---|
| `patient_manifest.csv` | one row per patient: MRN, ADT start, overall `risk_group` (`top`, `bottom`, or `top and bottom`), risk score / rank percentile / top-or-bottom-decile flag at each landmark, AVPC / platinum / NEPC event flag and days from ADT start, number of pathology notes |
| `pathology_reports.tsv` | one row per pathology report (all dates): every manifest column plus the note metadata (`RPT_ID`, `EVENT_DATE`, `PROC_DESC`, `RPT_TYPE`, `FILE`, days from ADT start) and `RPT_TEXT` |
| `pathology_term_enrichment.csv` | terms overrepresented in the top / bottom decile's pre-landmark pathology reports vs the middle 80% at each landmark (aggregate counts only) |
| `pathology_topics.csv` | optional: BERTopic topics fit on the same pre-landmark reports (topic size and top words only; no report text) |
| `pathology_topic_enrichment.csv` | optional: topics overrepresented in the top / bottom decile vs the middle 80% at each landmark (aggregate counts only) |

Output folder: `<COMPASS data root>/elio_data_requests/high_vs_low_risk_transfer_pxs/`. **The manifest and report files contain PHI**
(MRNs, report text); they stay under the protected data root and are never copied into the repository.

Run notebook 04 first: this reads its protected `all_models/transfer_scores.csv`. The risk score does not depend on the
evaluated endpoint, so the platinum rows are used. Deciles are taken among patients with finite numeric scores at that landmark.
Missing, nonnumeric, or infinite scores are excluded from cutoffs and ranks and labeled `not scored` in the manifest.
A patient is flagged `top` when their score is at or above the landmark's 90th percentile and `bottom` when at or below
its 10th. Ties at a cutoff are included, so groups may exceed 10%. There is no middle group: a landmark where the patient is in
neither decile has a blank `decile_group_lm<landmark>`. Extraction stops if a landmark has no valid scores
or its two cutoffs coincide, because that landmark cannot separate high- and low-risk groups.

## Configuration

In [ ]:
import os, sys
from pathlib import Path

sys.path.insert(0, "../../pipeline")
import compass_pipeline as cp

PROJECT_ROOT = cp.PROJECT_ROOT
for _p in (str(PROJECT_ROOT),):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd

from survival_common import federated_endpoint_transfer as fet
from survival_common import federated_scoring as fs

DATA_ROOT = Path(cp._PROFILE_OUTPUT_ROOT)
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(DATA_ROOT / "rhino_data_v2")))

# >>> Edit <<<
SCORES_PATH = RHINO_ROOT / "federated_transfer_adt_no_DFCI" / "all_models" / "transfer_scores.csv"   # notebook 04
DATA_PATH   = DATA_ROOT / "longitudinal_prediction_data_adt.csv"                                      # ADT anchor + endpoints
PATHOLOGY_PATH = Path(os.environ.get(
    "PROFILE_PATHOLOGY_NOTES",
    "/data/gusev/USERS/jpconnor/data/PROFILE_DATA/CLINICAL_NOTES/PATHOLOGY_NOTES.parquet"))
OUTPUT_DIR = DATA_ROOT / "elio_data_requests" / "high_vs_low_risk_transfer_pxs"

MODEL_FAMILY = "xgboost_cox"
SCORE_ENDPOINT = "platinum"        # the model is platinum-trained; risk_score is identical across endpoints
LANDMARKS = (0, 90, 180)
DECILE = 0.10                      # top and bottom 10% at each landmark
ENDPOINTS = ("avpc", "platinum", "nepc")

# Pathology term enrichment (top / bottom decile vs middle 80%)
TERM_MIN_PATIENTS = 5              # a term must appear in >= this many patients at the landmark
TERM_NGRAMS = (1, 2)               # unigrams and bigrams
TERM_FDR = 0.05                    # Benjamini-Hochberg threshold for the displayed tables
TERM_TOP_N = 25                    # terms shown per landmark x group

# Optional BERTopic on the same reports, from precomputed Clinical ModernBERT embeddings (read-only, from the
# clinical_text_embedding_project's knit_embeddings.py). The section skips if packages or files are missing.
CTEP_PROCESSED_DIR = Path(os.environ.get(
    "CTEP_DATA_PATH", "/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/"
)) / "batched_datasets" / "processed_datasets"
EMBED_METADATA_PATH = CTEP_PROCESSED_DIR / "full_clinical_notes_embeddings_metadata.parquet"
EMBED_ARRAY_PATH = CTEP_PROCESSED_DIR / "full_clinical_notes_embeddings_as_array.npy.zst"
TOPIC_MIN_SIZE = 30                # HDBSCAN min_cluster_size, in reports
TOPIC_SEED = 42                    # UMAP random_state

## Risk scores and deciles

One XGBoost full-model score per patient and landmark. The age-only baseline and any other family are dropped, and the
selection must leave exactly one row per patient and landmark.

In [ ]:
scores = pd.read_csv(SCORES_PATH, low_memory=False)
print(scores.groupby(["model_family", "config"]).size())

sel = scores[(scores["model_family"] == MODEL_FAMILY) & (scores["config"] != "baseline")
             & (scores["endpoint"] == SCORE_ENDPOINT) & (scores["landmark_days"].isin(LANDMARKS))]
sel = sel[[fs.ID_COL, "landmark_days", "risk_score"]]
dup = sel.duplicated([fs.ID_COL, "landmark_days"], keep=False)
assert not dup.any(), f"{dup.sum()} rows: more than one {MODEL_FAMILY} model/arm per patient and landmark; narrow the filter above"
assert set(sel["landmark_days"]) == set(LANDMARKS), f"landmarks found: {sorted(sel['landmark_days'].unique())}"

def decile_flags(g):
    if not 0 < DECILE < 0.5:
        raise ValueError("DECILE must be between 0 and 0.5 (exclusive).")
    g = g.copy()
    risk = pd.to_numeric(g["risk_score"], errors="coerce").astype(float)
    risk = risk.where(np.isfinite(risk))
    valid = risk.notna()
    landmark = g["landmark_days"].iloc[0]
    if not valid.any():
        raise ValueError(f"Landmark {landmark}: no finite numeric risk scores; extraction stopped.")
    lo, hi = risk[valid].quantile([DECILE, 1 - DECILE])
    if lo >= hi:
        raise ValueError(
            f"Landmark {landmark}: bottom and top cutoffs coincide; "
            "cannot separate high- and low-risk groups. Extraction stopped."
        )
    g["risk_score"] = risk
    g["risk_percentile"] = risk.rank(pct=True, method="average")
    # Only the two extreme deciles are labelled; everything in between stays blank.
    g["decile_group"] = pd.Series(pd.NA, index=g.index, dtype="string")
    g.loc[risk >= hi, "decile_group"] = "top"
    g.loc[risk <= lo, "decile_group"] = "bottom"
    g.loc[~valid, "decile_group"] = "not scored"
    g.attrs.update(lo=lo, hi=hi)
    return g

by_lm = {lm: decile_flags(g) for lm, g in sel.groupby("landmark_days")}
cutoffs = pd.DataFrame({lm: {"n_scored": int(g["risk_score"].notna().sum()),
                            "n_not_scored": int(g["risk_score"].isna().sum()),
                            "bottom_cutoff": g.attrs["lo"], "top_cutoff": g.attrs["hi"],
                            "n_top": int((g["decile_group"] == "top").sum()),
                            "n_bottom": int((g["decile_group"] == "bottom").sum())} for lm, g in by_lm.items()}).T
cutoffs

In [ ]:
wide = None
for lm, g in by_lm.items():
    part = g.rename(columns={"risk_score": f"risk_score_lm{lm}", "risk_percentile": f"risk_percentile_lm{lm}",
                             "decile_group": f"decile_group_lm{lm}"}).drop(columns="landmark_days")
    wide = part if wide is None else wide.merge(part, on=fs.ID_COL, how="outer")

group_cols = [f"decile_group_lm{lm}" for lm in LANDMARKS]
extreme = wide[group_cols].isin(["top", "bottom"]).any(axis=1)
cohort = wide[extreme].rename(columns={fs.ID_COL: "DFCI_MRN"}).copy()
for lm in LANDMARKS:
    col = f"decile_group_lm{lm}"
    cohort[f"top_decile_lm{lm}"] = cohort[col].eq("top").fillna(False).astype(bool)
    cohort[f"bottom_decile_lm{lm}"] = cohort[col].eq("bottom").fillna(False).astype(bool)
    # Absent from this landmark's score file -> not scored; scored but in neither decile stays blank.
    cohort.loc[cohort[f"risk_score_lm{lm}"].isna(), col] = "not scored"
is_top = cohort[[f"top_decile_lm{lm}" for lm in LANDMARKS]].any(axis=1)
is_bottom = cohort[[f"bottom_decile_lm{lm}" for lm in LANDMARKS]].any(axis=1)
cohort["risk_group"] = np.select([is_top & is_bottom, is_top], ["top and bottom", "top"], default="bottom")
print(f"{len(cohort):,} patients top/bottom decile at >=1 landmark (of {len(wide):,} patients in the score file)")
print(cohort["risk_group"].value_counts().to_string())

## ADT start and endpoints

ADT start is `TREATMENT_ANCHOR_DATE`, the anchor the landmarks are measured from. Each endpoint's date is used only when
its indicator is 1 (the date columns are back-filled for non-events upstream). `days_to_<endpoint>` counts from ADT start
and can be negative if the event was dated before ADT.

In [ ]:
def naive_dt(s):
    """Parse to datetime and drop any timezone (keeps wall-clock time) so note and ADT dates are comparable."""
    s = pd.to_datetime(s, errors="coerce", format="mixed", utc=False)
    if getattr(s.dt, "tz", None) is not None:
        s = s.dt.tz_localize(None)
    return s

local = fet.read_local_frame(DATA_PATH, ENDPOINTS)
cols = [fet.LOCAL_ID_COL, "TREATMENT_ANCHOR_DATE"] + [c for ep in ENDPOINTS for c in fet.ENDPOINT_SOURCES[ep]]
pt = local[cols].drop_duplicates(fet.LOCAL_ID_COL).copy()
pt["adt_start_datetime"] = naive_dt(pt["TREATMENT_ANCHOR_DATE"])
for ep in ENDPOINTS:
    ind, dt = fet.ENDPOINT_SOURCES[ep]
    event = pd.to_numeric(pt[ind], errors="coerce").fillna(0).eq(1)
    date = naive_dt(pt[dt]).where(event)
    pt[f"{ep}_event"] = event
    pt[f"{ep}_event_date"] = date
    pt[f"days_adt_to_{ep}"] = (date - pt["adt_start_datetime"]).dt.days
keep = [fet.LOCAL_ID_COL, "adt_start_datetime"] + [c for ep in ENDPOINTS for c in
        (f"{ep}_event", f"{ep}_event_date", f"days_adt_to_{ep}")]
pt = pt[keep].rename(columns={fet.LOCAL_ID_COL: "DFCI_MRN"})
pt["DFCI_MRN"] = pd.to_numeric(pt["DFCI_MRN"], errors="coerce").astype("Int64")
cohort["DFCI_MRN"] = pd.to_numeric(cohort["DFCI_MRN"], errors="coerce").astype("Int64")

manifest = cohort.merge(pt, on="DFCI_MRN", how="left", validate="one_to_one")
assert manifest["adt_start_datetime"].notna().all(), "scored patients missing an ADT start in the local frame"
manifest[[f"{ep}_event" for ep in ENDPOINTS]].sum()

## Pathology reports

Every pathology report on file for these patients, at any date (including after ADT start and after any event). Report
dates relative to ADT start are given so the request can be restricted downstream.

In [ ]:
import polars as pl

notes = (pl.scan_parquet(PATHOLOGY_PATH)
         .filter(pl.col("DFCI_MRN").cast(pl.Int64, strict=False).is_in(manifest["DFCI_MRN"].dropna().astype("int64").tolist()))
         .collect().to_pandas())
notes["DFCI_MRN"] = pd.to_numeric(notes["DFCI_MRN"], errors="coerce").astype("Int64")
notes["EVENT_DATE"] = naive_dt(notes["EVENT_DATE"])
print(f"{len(notes):,} pathology reports for {notes['DFCI_MRN'].nunique():,} of {len(manifest):,} patients")

counts = notes.groupby("DFCI_MRN").size().rename("n_pathology_notes")
manifest = manifest.merge(counts, on="DFCI_MRN", how="left")
manifest["n_pathology_notes"] = manifest["n_pathology_notes"].fillna(0).astype(int)
manifest = manifest.sort_values("DFCI_MRN").reset_index(drop=True)

# Column order: identity, ADT start, overall top/bottom group, per-landmark score / percentile / flags, endpoints, note count.
lm_cols = [c for lm in LANDMARKS for c in (f"risk_score_lm{lm}", f"risk_percentile_lm{lm}", f"decile_group_lm{lm}",
                                          f"top_decile_lm{lm}", f"bottom_decile_lm{lm}")]
ep_cols = [c for ep in ENDPOINTS for c in (f"{ep}_event", f"days_adt_to_{ep}", f"{ep}_event_date")]
manifest = manifest[["DFCI_MRN", "adt_start_datetime", "risk_group"] + lm_cols + ep_cols + ["n_pathology_notes"]]
print(f"Manifest ready: {len(manifest):,} patients; "
      f"{manifest['n_pathology_notes'].eq(0).sum():,} without pathology reports")

## Write

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT_DIR / "patient_manifest.csv"
manifest.to_csv(manifest_path, index=False)

note_cols = [c for c in ("RPT_ID", "EVENT_DATE", "PROC_DESC", "RPT_TYPE", "FILE") if c in notes.columns]
rep = notes[["DFCI_MRN"] + note_cols + ["RPT_TEXT"]].rename(columns={
    "EVENT_DATE": "note_date", **{c: f"note_{c.lower()}" for c in note_cols if c != "EVENT_DATE"}})
rep = manifest.merge(rep, on="DFCI_MRN", how="inner", validate="one_to_many")
rep["note_days_from_adt_start"] = (rep["note_date"] - rep["adt_start_datetime"]).dt.days
text_col = rep.pop("RPT_TEXT").rename("note_text")
rep = pd.concat([rep, text_col], axis=1).sort_values(["DFCI_MRN", "note_date"], kind="stable")
# Report bodies carry tabs and newlines; pandas quotes such fields, which any TSV reader honours.
tsv_path = OUTPUT_DIR / "pathology_reports.tsv"
rep.to_csv(tsv_path, sep="\t", index=False)

check = pd.read_csv(tsv_path, sep="\t", low_memory=False)
assert len(check) == len(rep) == int(manifest["n_pathology_notes"].sum())
print(f"wrote {manifest_path} ({len(manifest):,} patients)")
print(f"wrote {tsv_path} ({len(rep):,} reports; round-trip row count verified)")

## Pathology term enrichment: top / bottom decile vs middle 80%

At each landmark, patients fall into three groups by their score at that landmark: **top** decile, **bottom** decile, and
**middle** (scored, in neither decile; ties at a cutoff stay in the extreme groups, so this is at most 80%). The middle
group is the control. Each patient's group can differ between landmarks.

- **Reports:** every scored patient's pathology reports dated on or before ADT start + landmark days, i.e. what was on
  file when the patient was scored. Later reports (including post-progression pathology) and undated reports are left out.
  Patients with no report in the window drop out of that landmark's denominators.
- **Terms:** lowercased letter-only unigrams and bigrams (numbers, dates and IDs drop out), English stop words removed,
  counted within each report so bigrams never span two reports. A term is kept if it appears in at least
  `TERM_MIN_PATIENTS` patients at the landmark, which also keeps patient-specific strings such as names out.
- **Test:** a patient "has" a term if any of their in-window reports contains it. For top vs middle and bottom vs middle,
  one-sided Fisher's exact test for overrepresentation, Benjamini-Hochberg q-values within each contrast, and the odds ratio
  with a +0.5 correction. Mean patient-level TF-IDF (sublinear term frequency over each patient's pooled counts) is reported
  for each group as a descriptive column.

In [ ]:
import polars as pl

TERM_GROUPS = ("top", "bottom")
CONTROL_GROUP = "middle"

# Per-landmark group for every scored patient; a blank decile group means scored but in neither decile.
groups_by_lm = {}
for lm, g in by_lm.items():
    ids = pd.Index(pd.to_numeric(g[fs.ID_COL], errors="coerce").astype("Int64"), name="DFCI_MRN")
    grp = pd.Series(g["decile_group"].fillna(CONTROL_GROUP).to_numpy(), index=ids, name="group")
    groups_by_lm[lm] = grp[grp.ne("not scored")]

scored_ids = sorted({int(i) for grp in groups_by_lm.values() for i in grp.index.dropna()})
# RPT_ID (when present) links reports to the optional BERTopic embeddings; term enrichment does not use it.
note_cols = ["DFCI_MRN", "EVENT_DATE", "RPT_TEXT"]
if "RPT_ID" in pl.scan_parquet(PATHOLOGY_PATH).collect_schema().names():
    note_cols.append("RPT_ID")
all_notes = (pl.scan_parquet(PATHOLOGY_PATH)
             .select(note_cols)
             .filter(pl.col("DFCI_MRN").cast(pl.Int64, strict=False).is_in(scored_ids))
             .collect().to_pandas())
all_notes["DFCI_MRN"] = pd.to_numeric(all_notes["DFCI_MRN"], errors="coerce").astype("Int64")
all_notes["note_day"] = naive_dt(all_notes["EVENT_DATE"]).dt.normalize()
all_notes["RPT_TEXT"] = all_notes["RPT_TEXT"].fillna("").astype(str)
all_notes = all_notes.merge(pt[["DFCI_MRN", "adt_start_datetime"]], on="DFCI_MRN", how="left", validate="many_to_one")
adt_day = all_notes["adt_start_datetime"].dt.normalize()

# Reports on or before ADT start + landmark; NaT dates (undated report or no ADT start) compare False and drop out.
reports_by_lm, size_rows = {}, []
for lm, grp in groups_by_lm.items():
    in_window = (all_notes["note_day"].le(adt_day + np.timedelta64(int(lm), "D"))
                 & all_notes["DFCI_MRN"].isin(grp.index))
    rep_lm = all_notes.loc[in_window, ["DFCI_MRN", "RPT_TEXT"] + note_cols[3:]].reset_index(drop=True)
    reports_by_lm[lm] = rep_lm
    has_report = grp.index.isin(rep_lm["DFCI_MRN"])
    for name in (*TERM_GROUPS, CONTROL_GROUP):
        in_group = grp.eq(name).to_numpy()
        size_rows.append({"landmark": lm, "group": name, "patients_scored": int(in_group.sum()),
                          "patients_with_report": int((in_group & has_report).sum()),
                          "reports": int(rep_lm["DFCI_MRN"].isin(grp.index[in_group]).sum())})

print(f"{all_notes['note_day'].isna().sum():,} undated pathology reports excluded; "
      f"{all_notes['adt_start_datetime'].isna().sum():,} reports from patients without an ADT start excluded")
group_sizes = pd.DataFrame(size_rows).set_index(["landmark", "group"])
group_sizes

In [ ]:
from scipy import sparse
from scipy.stats import hypergeom
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

TOKEN_PATTERN = r"(?u)\b[a-z][a-z]{2,}\b"   # applied after lowercasing: letters only, >= 3 characters
ENRICHMENT_COLS = ["landmark", "group", "term", "n_group", "n_control", "group_with_term", "control_with_term",
                   "pct_group", "pct_control", "odds_ratio", "p_value", "q_value",
                   "mean_tfidf_group", "mean_tfidf_control"]


def bh_qvalues(p):
    """Benjamini-Hochberg q-values (scipy on the cluster predates false_discovery_control)."""
    p = np.asarray(p, dtype=float)
    if p.size == 0:
        return p
    order = np.argsort(p)
    ranked = p[order] * p.size / np.arange(1, p.size + 1)
    q = np.minimum(np.minimum.accumulate(ranked[::-1])[::-1], 1.0)
    out = np.empty_like(q)
    out[order] = q
    return out


def patient_term_matrices(reports, ids):
    """Patient x term presence and TF-IDF, with terms counted within each report and summed per patient."""
    vec = CountVectorizer(lowercase=True, stop_words="english", token_pattern=TOKEN_PATTERN,
                          ngram_range=TERM_NGRAMS, min_df=TERM_MIN_PATIENTS)
    try:
        # Report-level min_df is a safe prefilter: a term in >= k patients is in >= k reports.
        counts = vec.fit_transform(reports["RPT_TEXT"].to_numpy())
    except ValueError:  # no term survives (or fewer reports than TERM_MIN_PATIENTS)
        return np.array([], dtype=object), None, None
    rows = ids.get_indexer(reports["DFCI_MRN"])
    to_patient = sparse.csr_matrix((np.ones(len(rows)), (rows, np.arange(len(rows)))),
                                   shape=(len(ids), len(rows)))
    patient_counts = (to_patient @ counts).tocsc()
    present = (patient_counts > 0).astype(np.int64)
    keep = np.flatnonzero(np.asarray(present.sum(axis=0)).ravel() >= TERM_MIN_PATIENTS)
    if not keep.size:
        return np.array([], dtype=object), None, None
    patient_counts, present = patient_counts[:, keep], present[:, keep]
    tfidf = TfidfTransformer(sublinear_tf=True).fit_transform(patient_counts)
    return vec.get_feature_names_out()[keep], present.tocsr(), tfidf.tocsr()


def fisher_enrichment(present, label, features, landmark):
    """One-sided Fisher overrepresentation of each feature column in each of TERM_GROUPS vs CONTROL_GROUP.

    present: patients x features 0/1 sparse matrix; label: each patient's group. Rows come out grouped by
    TERM_GROUPS order, len(features) rows per group with a non-empty group and control.
    """
    control = label == CONTROL_GROUP
    n0 = int(control.sum())
    c = np.asarray(present[control].sum(axis=0)).ravel()
    out = []
    for name in TERM_GROUPS:
        in_group = label == name
        n1 = int(in_group.sum())
        if n1 == 0 or n0 == 0:
            continue
        a = np.asarray(present[in_group].sum(axis=0)).ravel()
        p = hypergeom.sf(a - 1, n1 + n0, a + c, n1)       # P(X >= a): one-sided Fisher
        out.append(pd.DataFrame({
            "landmark": landmark, "group": name, "feature": features, "n_group": n1, "n_control": n0,
            "group_with_feature": a, "control_with_feature": c, "pct_group": a / n1, "pct_control": c / n0,
            "odds_ratio": ((a + 0.5) * (n0 - c + 0.5)) / ((n1 - a + 0.5) * (c + 0.5)),
            "p_value": p, "q_value": bh_qvalues(p),
        }))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


def term_enrichment(reports, groups, landmark):
    """Fisher overrepresentation of each term in each of TERM_GROUPS vs CONTROL_GROUP at one landmark."""
    ids = pd.Index(sorted(reports["DFCI_MRN"].unique()), dtype="Int64")   # patients with >= 1 in-window report
    terms, present, tfidf = patient_term_matrices(reports, ids)
    if not len(terms):
        return pd.DataFrame(columns=ENRICHMENT_COLS)
    label = groups.reindex(ids).to_numpy()
    res = fisher_enrichment(present, label, terms, landmark)
    if res.empty:
        return pd.DataFrame(columns=ENRICHMENT_COLS)
    mean_tfidf = {name: np.asarray(tfidf[label == name].mean(axis=0)).ravel()
                  for name in res["group"].drop_duplicates()}
    res["mean_tfidf_group"] = np.concatenate(list(mean_tfidf.values()))
    res["mean_tfidf_control"] = np.tile(np.asarray(tfidf[label == CONTROL_GROUP].mean(axis=0)).ravel(),
                                        len(mean_tfidf))
    return res.rename(columns={"feature": "term", "group_with_feature": "group_with_term",
                               "control_with_feature": "control_with_term"})


enrichment = pd.concat(
    [term_enrichment(reports_by_lm[lm], groups_by_lm[lm], lm) for lm in LANDMARKS if lm in reports_by_lm],
    ignore_index=True,
).reindex(columns=ENRICHMENT_COLS)
enrichment = enrichment.sort_values(["landmark", "group", "p_value", "odds_ratio"],
                                    ascending=[True, True, True, False], ignore_index=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
enrichment_path = OUTPUT_DIR / "pathology_term_enrichment.csv"
enrichment.to_csv(enrichment_path, index=False)
print(f"wrote {enrichment_path} ({len(enrichment):,} term x contrast rows)")

In [ ]:
try:
    from IPython.display import display
except ImportError:  # plain-Python runs
    display = print

show_cols = ["term", "group_with_term", "n_group", "pct_group", "control_with_term", "n_control", "pct_control",
             "odds_ratio", "q_value", "mean_tfidf_group", "mean_tfidf_control"]
for lm in LANDMARKS:
    for name in TERM_GROUPS:
        sub = enrichment[(enrichment["landmark"] == lm) & (enrichment["group"] == name)]
        title = f"Landmark {lm} · {name} decile vs middle"
        if sub.empty:
            print(f"\n{title}: no terms (too few patients or reports)")
            continue
        sig = sub[sub["q_value"] < TERM_FDR]
        note = f"q < {TERM_FDR}: {len(sig):,} terms"
        if sig.empty:
            sig, note = sub, f"none at q < {TERM_FDR}; top {TERM_TOP_N} by p shown"
        top = sig.sort_values(["p_value", "odds_ratio"], ascending=[True, False]).head(TERM_TOP_N)
        print(f"\n{title} ({note})")
        display(top[show_cols].round(3).reset_index(drop=True))

## Optional: BERTopic topics in the same reports

Groups, reports and the date window are the same as above. Instead of single terms, reports are clustered into topics,
and topics are tested for overrepresentation the same way.

- **Embeddings:** precomputed, mean-pooled Clinical ModernBERT embeddings of each pathology report, read (never written)
  from the clinical_text_embedding_project (`full_clinical_notes_embeddings_as_array.npy.zst` and its metadata parquet),
  joined on `RPT_ID`. No language model is loaded and nothing is downloaded. Reports outside that project's cohort have
  no embedding; the coverage table shows how many patients each contrast keeps.
- **Model:** one BERTopic fit on the union of in-window, embedded reports across landmarks (each report once), so topic
  numbers mean the same thing at every landmark. UMAP (5 dimensions, cosine, `TOPIC_SEED`) then HDBSCAN
  (`TOPIC_MIN_SIZE` reports). Topic words use only the terms that appear in at least `TERM_MIN_PATIENTS` patients.
- **Test:** a patient "has" a topic if any of their in-window embedded reports is assigned to it. Outlier reports
  (topic -1) are not a feature, but their patients stay in the denominators. Denominators are patients with at least one
  embedded in-window report. Topics in fewer than `TERM_MIN_PATIENTS` patients at a landmark are dropped there.
  Same one-sided Fisher test, BH q-values and odds ratio as the terms.
- **PHI:** BERTopic keeps representative documents (raw report text) inside the fitted model. They are dropped from
  every table and the model is never saved; only topic sizes and top words are written.

The section needs `bertopic`, `umap-learn`, `hdbscan` and `zstandard` (commented out in `requirements.txt`). Without
them, or without the embedding files, it prints why it skipped and writes nothing; the term enrichment above is unaffected.

In [ ]:
import importlib
import io

TOPIC_PACKAGES = {"zstandard": "zstandard", "bertopic": "bertopic", "umap": "umap-learn", "hdbscan": "hdbscan"}


def optional_import(name):
    try:
        return importlib.import_module(name)
    except Exception:   # ImportError, or a broken install (e.g. numba) that fails on import
        return None


def rpt_key(values):
    """RPT_ID as a string join key; whole-number IDs lose any float formatting ("12.0" -> "12")."""
    s = pd.Series(values).reset_index(drop=True)
    num = pd.to_numeric(s, errors="coerce")
    whole = num.notna() & np.isfinite(num) & (num % 1 == 0)
    key = s.astype("string").str.strip()
    key[whole] = num[whole].astype("int64").astype(str)
    return key.where(s.notna())


def _read_exact(fh, n):
    parts, got = [], 0
    while got < n:
        chunk = fh.read(n - got)
        if not chunk:
            raise ValueError("embedding array ended before its header's shape")
        parts.append(chunk)
        got += len(chunk)
    return b"".join(parts)


def read_embedding_rows(path, rows, zstd, chunk_rows=16384):
    """Rows `rows` (sorted, unique) of a zstd-compressed 2-D .npy array, as float32, in one streaming pass.

    The full array (every note type for the embedding cohort) is never held in memory: it is decompressed in
    chunk_rows blocks and only the requested rows are kept.
    """
    rows = np.asarray(rows, dtype=np.int64)
    with open(path, "rb") as fh, zstd.ZstdDecompressor().stream_reader(fh) as raw:
        version = np.lib.format.read_magic(raw)
        readers = {(1, 0): np.lib.format.read_array_header_1_0, (2, 0): np.lib.format.read_array_header_2_0}
        if version not in readers:
            raise ValueError(f"unsupported .npy version {version}")
        shape, fortran_order, dtype = readers[version](raw)
        if fortran_order or len(shape) != 2 or dtype.hasobject:
            raise ValueError(f"expected a C-order 2-D numeric array, got shape {shape}")
        n_rows, dim = shape
        if rows.size and (rows[0] < 0 or rows[-1] >= n_rows):
            raise ValueError(f"EMBEDDING_INDEX outside the array's {n_rows:,} rows")
        out = np.empty((rows.size, dim), dtype=np.float32)
        start, stop = 0, int(rows[-1]) + 1 if rows.size else 0
        while start < stop:
            k = min(chunk_rows, stop - start)
            block = np.frombuffer(_read_exact(raw, k * dim * dtype.itemsize), dtype=dtype).reshape(k, dim)
            lo, hi = np.searchsorted(rows, [start, start + k])
            out[lo:hi] = block[rows[lo:hi] - start]
            start += k
    return out


topic_mods = {name: optional_import(name) for name in TOPIC_PACKAGES}
missing = [pip for name, pip in TOPIC_PACKAGES.items() if topic_mods[name] is None]
topic_skip = None
if missing:
    topic_skip = f"packages not installed: {', '.join(missing)}"
elif not (EMBED_METADATA_PATH.exists() and EMBED_ARRAY_PATH.exists()):
    topic_skip = f"precomputed embeddings not found under {EMBED_ARRAY_PATH.parent}"
elif "RPT_ID" not in all_notes:
    topic_skip = "pathology notes have no RPT_ID to join the embeddings on"

topic_docs = None
if topic_skip is None:
    # Each in-window report once, across landmarks (the windows only differ in which reports they include).
    topic_docs = (pd.concat(reports_by_lm.values(), ignore_index=True)
                  .assign(rpt_key=lambda d: rpt_key(d["RPT_ID"]).to_numpy())
                  .dropna(subset=["rpt_key"]).drop_duplicates("rpt_key", ignore_index=True))
    meta = (pl.scan_parquet(EMBED_METADATA_PATH)
            .filter(pl.col("NOTE_TYPE") == "Pathology")
            .select("RPT_ID", "EMBEDDING_INDEX")
            .collect().to_pandas())
    meta["rpt_key"] = rpt_key(meta["RPT_ID"]).to_numpy()
    meta = meta.dropna(subset=["rpt_key", "EMBEDDING_INDEX"]).drop_duplicates("rpt_key")
    n_window = len(topic_docs)
    topic_docs = topic_docs.merge(meta[["rpt_key", "EMBEDDING_INDEX"]], on="rpt_key", how="inner", validate="one_to_one")
    print(f"{len(topic_docs):,} of {n_window:,} in-window pathology reports have a precomputed embedding")
    if len(topic_docs) <= 2 * TOPIC_MIN_SIZE:
        topic_skip = f"only {len(topic_docs):,} embedded reports (need more than {2 * TOPIC_MIN_SIZE})"

if topic_skip is None:
    topic_docs = topic_docs.sort_values("EMBEDDING_INDEX", ignore_index=True)
    rows = topic_docs["EMBEDDING_INDEX"].astype("int64").to_numpy()
    embeddings = read_embedding_rows(EMBED_ARRAY_PATH, rows, topic_mods["zstandard"])
    # Topic words come only from terms in >= TERM_MIN_PATIENTS patients, the same rule as the term enrichment.
    vocab, _, _ = patient_term_matrices(topic_docs, pd.Index(sorted(topic_docs["DFCI_MRN"].unique()), dtype="Int64"))
    if not len(vocab):
        topic_skip = f"no term appears in >= {TERM_MIN_PATIENTS} patients' embedded reports"

if topic_skip is None:
    topic_model = topic_mods["bertopic"].BERTopic(
        embedding_model=None,   # embeddings are passed in: no backend is selected or downloaded
        umap_model=topic_mods["umap"].UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine",
                                           random_state=TOPIC_SEED),
        hdbscan_model=topic_mods["hdbscan"].HDBSCAN(min_cluster_size=TOPIC_MIN_SIZE, metric="euclidean",
                                                    cluster_selection_method="eom", prediction_data=False),
        vectorizer_model=CountVectorizer(vocabulary=list(vocab), ngram_range=TERM_NGRAMS, lowercase=True,
                                         token_pattern=TOKEN_PATTERN),
        calculate_probabilities=False,
        verbose=False,
    )
    assigned, _ = topic_model.fit_transform(topic_docs["RPT_TEXT"].tolist(), embeddings=embeddings)
    topic_docs["topic"] = np.asarray(assigned, dtype=int)
    info = topic_model.get_topic_info()
    words = {int(t): " ".join(w for w, _ in (topic_model.get_topic(t) or [])) for t in info["Topic"]}
    del topic_model, info, embeddings   # the model holds representative report text: never saved or shown
    topic_table = (topic_docs.groupby("topic")
                   .agg(n_reports=("rpt_key", "size"), n_patients=("DFCI_MRN", "nunique"))
                   .reset_index())
    topic_table["top_words"] = topic_table["topic"].map(words).fillna("")
    topic_table.loc[topic_table["topic"].eq(-1), "top_words"] = "(outliers: no topic)"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    topics_path = OUTPUT_DIR / "pathology_topics.csv"
    topic_table.to_csv(topics_path, index=False)
    print(f"wrote {topics_path} ({topic_table['topic'].ge(0).sum():,} topics, "
          f"{topic_table.loc[topic_table['topic'].eq(-1), 'n_reports'].sum():,} outlier reports)")

if topic_skip is not None:
    topic_docs = None
    print(f"BERTopic section skipped ({topic_skip}); no topic files written.")

In [ ]:
TOPIC_COLS = ["landmark", "group", "topic", "top_words", "n_group", "n_control", "group_with_topic",
              "control_with_topic", "pct_group", "pct_control", "odds_ratio", "p_value", "q_value"]


def topic_enrichment(docs, groups, landmark):
    """Fisher overrepresentation of each topic in each of TERM_GROUPS vs CONTROL_GROUP at one landmark."""
    ids = pd.Index(sorted(docs["DFCI_MRN"].unique()), dtype="Int64")   # patients with >= 1 embedded report
    topics = np.array(sorted(t for t in docs["topic"].unique() if t >= 0))
    hits = docs[docs["topic"].ge(0)]
    present = sparse.csr_matrix(
        (np.ones(len(hits)), (ids.get_indexer(hits["DFCI_MRN"]), np.searchsorted(topics, hits["topic"]))),
        shape=(len(ids), len(topics)))
    present = (present > 0).astype(np.int64).tocsc()
    keep = np.flatnonzero(np.asarray(present.sum(axis=0)).ravel() >= TERM_MIN_PATIENTS)
    if not keep.size:
        return pd.DataFrame(columns=TOPIC_COLS)
    res = fisher_enrichment(present[:, keep].tocsr(), groups.reindex(ids).to_numpy(), topics[keep], landmark)
    if res.empty:
        return pd.DataFrame(columns=TOPIC_COLS)
    res = res.rename(columns={"feature": "topic", "group_with_feature": "group_with_topic",
                              "control_with_feature": "control_with_topic"})
    return res.assign(top_words=res["topic"].map(topic_table.set_index("topic")["top_words"]))


if topic_docs is None:
    print("BERTopic section skipped; see the previous cell.")
else:
    topic_frames, coverage_rows = [], []
    keys = topic_docs.set_index("rpt_key")
    for lm in LANDMARKS:
        if lm not in reports_by_lm:
            continue
        grp = groups_by_lm[lm]
        in_window = rpt_key(reports_by_lm[lm]["RPT_ID"]).dropna().drop_duplicates()
        docs_lm = keys.loc[keys.index.intersection(pd.Index(in_window)), ["DFCI_MRN", "topic"]].reset_index()
        topic_frames.append(topic_enrichment(docs_lm, grp, lm))
        for name in (*TERM_GROUPS, CONTROL_GROUP):
            members = grp.index[grp.eq(name).to_numpy()]
            in_group = docs_lm["DFCI_MRN"].isin(members)
            coverage_rows.append({"landmark": lm, "group": name,
                                  "patients_with_report": group_sizes.loc[(lm, name), "patients_with_report"],
                                  "patients_with_embedded_report": int(docs_lm.loc[in_group, "DFCI_MRN"].nunique()),
                                  "embedded_reports": int(in_group.sum()),
                                  "outlier_reports": int((in_group & docs_lm["topic"].eq(-1)).sum())})
    topic_enrich = (pd.concat(topic_frames, ignore_index=True).reindex(columns=TOPIC_COLS)
                    .sort_values(["landmark", "group", "p_value", "odds_ratio"],
                                 ascending=[True, True, True, False], ignore_index=True))
    topic_enrich_path = OUTPUT_DIR / "pathology_topic_enrichment.csv"
    topic_enrich.to_csv(topic_enrich_path, index=False)
    print(f"wrote {topic_enrich_path} ({len(topic_enrich):,} topic x contrast rows)")

    topic_coverage = pd.DataFrame(coverage_rows).set_index(["landmark", "group"])
    display(topic_coverage)
    topic_show = ["topic", "top_words", "group_with_topic", "n_group", "pct_group", "control_with_topic",
                  "n_control", "pct_control", "odds_ratio", "q_value"]
    for lm in LANDMARKS:
        for name in TERM_GROUPS:
            sub = topic_enrich[(topic_enrich["landmark"] == lm) & (topic_enrich["group"] == name)]
            title = f"Landmark {lm} · {name} decile vs middle (topics)"
            if sub.empty:
                print(f"\n{title}: no topics (too few patients or reports)")
                continue
            print(f"\n{title} (q < {TERM_FDR}: {int((sub['q_value'] < TERM_FDR).sum()):,} topics)")
            display(sub[topic_show].head(TERM_TOP_N).round(3).reset_index(drop=True))